# 蒸馏（DINOv2 → 剪枝 YOLO）学习笔记

本 notebook 只注重「思路 + 重要命令 + 关键代码」，**不带运行输出**。每个流程按 **原理 / PowerShell 与 Bash / 重要代码 / 实验结论** 整理。

代码来自 `scripts/distill_dinov2_coco128.py`，结果来自实验 12、13 报告。

> 当前阶段结论：DINOv2 蒸馏没有提高 COCO128 上的峰值精度，但缓解了训练后期退化；P4 单尺度优于当前 P3/P4/P5 多尺度。

## 一、通用准备（先把概念和环境讲清楚）

### 1. 知识蒸馏到底是什么

**知识蒸馏（Knowledge Distillation, KD）**：训练学生时，除真实标签产生的任务损失外，再加入教师模型提供的监督。

常见三类蒸馏：

- **输出蒸馏**：对齐教师和学生的类别概率、目标框或检测头输出。
- **特征蒸馏**：对齐中间特征图。本项目采用这种方式。
- **关系蒸馏**：对齐样本、通道或空间位置之间的关系。

本项目不是传统“分类 logits + temperature”的软标签蒸馏。DINOv2 没有 YOLO 检测头，不能直接提供同结构的框和类别 logits，因此使用中间特征蒸馏。

总目标：

[
L_{total}=L_{YOLO}+lambda(t)L_{DINO}
]

其中 (lambda(t)) 在前两轮线性增大，避免训练一开始蒸馏梯度突然压过检测梯度。

### 2. 环境、数据和模型

核心依赖：

- `ultralytics`：加载、训练和验证 YOLO。
- `torch` / `torchvision`：模型、自动微分和插值。
- 官方 DINOv2 Torch Hub：加载冻结的 `dinov2_vits14`。
- CUDA GPU：当前实验使用 RTX 5060 Ti 16 GB。

学生模型是实验 05 贪心结构化剪枝后微调得到的 best：

`runs/prune/experiment05_greedy/20260908_161152/finetune/greedy/weights/best.pt`

数据为固定种子划分的 COCO128：102 张训练图、26 张验证图。

**重要命令：环境检查**

PowerShell：

~~~powershell
# 命令依据：README.md 第 62–80 行；环境版本来源于当前项目 .venv
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe -c "import torch, ultralytics; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('ultralytics', ultralytics.__version__)"
~~~

Linux Bash：

~~~bash
# 命令依据：README.md 第 62–80 行；先进入仓库根目录
.venv/bin/python -c "import torch, ultralytics; print('torch', torch.__version__); print('cuda', torch.cuda.is_available()); print('ultralytics', ultralytics.__version__)"
~~~

PowerShell 多行续写用反引号，Bash 用反斜杠 `\`。Windows 虚拟环境解释器通常在 `.venv\Scripts\python.exe`，Linux 在 `.venv/bin/python`。

In [ ]:
# 规范化学习示例，并非原脚本逐字复制
# 依据：scripts/distill_dinov2_coco128.py
#   ROOT/默认路径：第 34–40 行
#   学生模型加载：train_distill() 第 389–396 行
# 所属阶段：确定项目根目录、输入权重和数据，再加载学生模型

from pathlib import Path

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
student_weights = (
    PROJECT_ROOT
    / "runs/prune/experiment05_greedy/20260908_161152/finetune/greedy/weights/best.pt"
)
data_config = PROJECT_ROOT / "configs/coco128_split.yaml"

if not student_weights.is_file():
    raise FileNotFoundError(student_weights)
if not data_config.is_file():
    raise FileNotFoundError(data_config)

student_yolo = YOLO(str(student_weights))  # Ultralytics 高层入口
student_model = student_yolo.model  # 真正参与前向和训练的 PyTorch 模型

### 3. 整体数据流（老师问原理时先讲这一张）

~~~text
同一批输入图像
   │
   ├──→ 剪枝 YOLO 学生 ──→ 检测预测 ──→ L_YOLO（box + cls + dfl）
   │              │
   │              └──→ P4 或 P3/P4/P5 特征 ──→ 1×1 Conv ──┐
   │                                                        │ cosine distance
   └──→ 冻结 DINOv2 教师 ──→ patch token / 中间层特征 ──→ resize ──┘
                                                            │
                                           L_total = L_YOLO + λ(t)L_DINO
~~~

反向传播时：

- YOLO 学生和 1×1 投影层接收梯度。
- DINOv2 始终 `eval()` 且 `requires_grad=False`，不更新。
- 部署时删除教师、投影层和 hook，只保留 YOLO，因此推理结构与剪枝模型相同。

## 二、DINOv2 教师和特征对齐

### 4. 为什么选择 DINOv2 ViT-S/14

- DINOv2 是自监督视觉 Transformer，学习到较通用的视觉表示。
- ViT-S/14 的特征维度是 384，比 ViT-B/L 更适合本机快速验证。
- “14”表示 patch size 为 14。输入高宽应能被 14 整除。
- YOLO 输入仍是 640×640；仅教师支路插值到 **644×644**，因为 `644 / 14 = 46`。
- 教师产生 46×46 patch 网格，再插值到学生的 80×80、40×40 或 20×20。

注意：DINOv2 是通用表征教师，不是目标检测教师。它不直接提供类别和边界框监督，这是可能无法提高检测 best 的重要原因。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
# 对应：DINOFeatureDistiller.__init__() 第 107–124 行
# 所属阶段：创建并冻结教师，同时建立学生到教师通道数的投影层


class DINOFeatureDistiller(nn.Module):
    def __init__(
        self,
        student_channels: int,
        device: torch.device,
        dino_size: int,
        weight: float,
    ):
        super().__init__()
        if dino_size % 14:
            raise ValueError("--dino-size must be divisible by DINOv2 patch size 14")

        self.dino_size = dino_size
        self.base_weight = weight
        self.current_weight = 0.0

        self.teacher = (
            torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(device).eval()
        )
        for parameter in self.teacher.parameters():
            parameter.requires_grad_(False)

        self.projector = nn.Conv2d(
            student_channels,
            384,
            kernel_size=1,
            bias=False,
        ).to(device)
        nn.init.kaiming_normal_(
            self.projector.weight,
            mode="fan_out",
            nonlinearity="linear",
        )

        self.register_buffer(
            "mean",
            torch.tensor(DINO_MEAN, device=device).view(1, 3, 1, 1),
            persistent=False,
        )
        self.register_buffer(
            "std",
            torch.tensor(DINO_STD, device=device).view(1, 3, 1, 1),
            persistent=False,
        )

### 5. 为什么需要 1×1 Conv、插值和 cosine distance

教师、学生特征不能直接比较：

- YOLO P4 通道数为 128，DINOv2 ViT-S/14 为 384。
- YOLO P4 是 40×40，DINO patch 网格是 46×46。

处理步骤：

1. 可训练 1×1 Conv 把学生通道投影到 384。
2. bilinear interpolation 把教师空间尺寸改成学生尺寸。
3. 在通道维做 L2 normalize。
4. 使用 `1 - mean(cosine_similarity)`。

cosine 主要比较方向，避免 CNN 和 ViT 特征幅值不同造成干扰。但全图平均会蒸馏大量背景，而且 projector 也可能自己吸收一部分对齐任务，所以“DINO loss 下降”不必然等于“mAP 上升”。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
# 对应：DINOFeatureDistiller.teacher_feature()/loss() 第 126–140 行
# 输入：YOLO 已归一化到 0–1 的图像、学生 P4 特征
# 输出：一个标量 cosine distance；梯度只流向 projector 和学生模型


@torch.no_grad()
def teacher_feature(
    self, images: torch.Tensor, output_size: tuple[int, int]
) -> torch.Tensor:
    images = F.interpolate(
        images,
        size=(self.dino_size, self.dino_size),
        mode="bilinear",
        align_corners=False,
    )
    normalized = (images - self.mean) / self.std
    tokens = self.teacher.forward_features(normalized)["x_norm_patchtokens"]

    grid = self.dino_size // 14
    tokens = tokens.transpose(1, 2).reshape(
        images.shape[0],
        384,
        grid,
        grid,
    )
    return F.interpolate(
        tokens,
        size=output_size,
        mode="bilinear",
        align_corners=False,
    )


def loss(self, student_feature: torch.Tensor, images: torch.Tensor) -> torch.Tensor:
    teacher_feature = self.teacher_feature(images, student_feature.shape[-2:])
    projected_student = F.normalize(self.projector(student_feature), dim=1)
    normalized_teacher = F.normalize(teacher_feature, dim=1)
    return 1.0 - (projected_student * normalized_teacher).sum(dim=1).mean()

## 三、实验 12：P4 单尺度 DINOv2 蒸馏

### 6. 为什么先选 P4

YOLO 检测头接收三个尺度：

- P3：stride 8，80×80，偏小目标和局部纹理。
- P4：stride 16，40×40，空间细节与语义的折中。
- P5：stride 32，20×20，偏大目标和高级语义。

第一次实验只选 P4，可以减少变量、计算量和错误来源。DINOv2 最终层语义较强，P4 比 P3 更容易对齐语义，同时比 P5 保留更多空间信息。

In [ ]:
# 规范化学习摘录：调整了定义顺序，逻辑与原项目一致
# 出处：scripts/distill_dinov2_coco128.py
#   CaptureP4：第 188–200 行
#   _find_p4_layer()：第 264–270 行
#   attach_distillation()：第 281–299 行
# 所属阶段：定位 Detect 的 P4 输入层，并用 hook 保存该层输出


class CaptureP4:
    """每次 P4 完成前向后，将输出张量保存到共享字典。"""

    def __init__(self, features: dict[str, torch.Tensor]):
        self.features = features

    def __call__(
        self, _module: nn.Module, _inputs: tuple, output: torch.Tensor
    ) -> None:
        self.features["p4"] = output

    def __getstate__(self):
        # checkpoint 序列化前清掉可能仍连接计算图的张量
        self.features.clear()
        return {"features": self.features}


def _find_p4_layer(model: nn.Module) -> nn.Module:
    detect = model.model[-1]
    sources = getattr(detect, "f", None)
    if not isinstance(sources, (list, tuple)) or len(sources) < 2:
        raise RuntimeError("Could not identify Detect's P4 input layer")
    return model.model[sources[1]]


captured: dict[str, torch.Tensor] = {}
p4_layer = _find_p4_layer(student_model)
p4_layer.register_forward_hook(CaptureP4(captured))

# 后续训练前向后，P4 特征位于 captured["p4"]。
# 正式脚本还会先用一张 640×640 零张量探测实际通道数，再创建 projector。

### 7. 怎样把蒸馏损失加入 YOLO

Ultralytics 检测损失包含 box、cls、dfl。自定义模型先调用父类得到原检测损失，再追加 DINO loss：

[
L=L_{box}+L_{cls}+L_{dfl}+lambda(t)L_{DINO}
]

Ultralytics 的训练损失按 batch size 缩放，因此 DINO 分量也乘 batch size，保持口径一致。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
# 对应：DINOStudentModel.loss() 第 217–232 行
# 所属阶段：把检测损失和蒸馏损失交给 Ultralytics 统一求和


class DINOStudentModel(DetectionModel):
    def loss(self, batch: dict, preds=None):
        regular_loss, loss_items = super().loss(batch, preds)
        if not self.training:
            return regular_loss, loss_items

        student_feature = self._dino_student_features.get("p4")
        if student_feature is None:
            raise RuntimeError("P4 hook did not capture a student feature")

        dino_loss = self._dino_feature_distiller.loss(
            student_feature,
            batch["img"],
        )
        loss_items["dino_loss"] = dino_loss.detach()

        weight = self._dino_feature_distiller.current_weight
        batch_size = batch["img"].shape[0]
        scaled_dino_loss = dino_loss * weight * batch_size

        return torch.cat((regular_loss, scaled_dino_loss.reshape(1))), loss_items

### 8. 冻结教师、冻结 BN 和 warm-up

COCO128 只有 102 张训练图，BatchNorm 运行统计量容易被破坏，因此固定 BN。Ultralytics 在训练准备阶段可能重新开启浮点参数梯度，所以在构造优化器前和每次 `model.train()` 后都再次冻结。

蒸馏权重最终为 0.5，前 2 个 epoch 线性 warm-up：

- epoch 1：0.25
- epoch 2 及以后：0.50

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
#   freeze_batchnorm()/FrozenBNTrainer：第 73–104 行
#   warm-up 回调：train_distill() 第 398–408 行
# 所属阶段：每次切换训练模式后重新冻结教师和 BN，并逐轮设置蒸馏权重


def freeze_batchnorm(trainer: DetectionTrainer) -> None:
    for module in trainer.model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()
            for parameter in module.parameters():
                parameter.requires_grad_(False)


class FrozenBNTrainer(DetectionTrainer):
    def _freeze_runtime(self) -> None:
        distiller = getattr(self.model, "_dino_feature_distiller", None)
        if distiller is not None:
            distiller.teacher.eval()
            for parameter in distiller.teacher.parameters():
                parameter.requires_grad_(False)
        freeze_batchnorm(self)

    def build_optimizer(self, *args, **kwargs):
        self._freeze_runtime()
        return super().build_optimizer(*args, **kwargs)

    def _model_train(self):
        super()._model_train()
        self._freeze_runtime()


def prepare(trainer: FrozenBNTrainer) -> None:
    warmup_epochs = max(1, args.warmup_epochs)
    trainer.model._dino_feature_distiller.current_weight = args.weight / warmup_epochs
    trainer._freeze_runtime()


def set_weight(trainer: FrozenBNTrainer) -> None:
    warmup_epochs = max(1, args.warmup_epochs)
    fraction = min(1.0, (trainer.epoch + 1) / warmup_epochs)
    trainer.model._dino_feature_distiller.current_weight = args.weight * fraction
    trainer._freeze_runtime()


trainer.callbacks["on_pretrain_routine_end"].append(prepare)
trainer.callbacks["on_train_epoch_start"].append(set_weight)

### 9. 实验 12 运行命令

PowerShell：

~~~powershell
# 入口出处：scripts/distill_dinov2_coco128.py -> main()（第 510–593 行）
# 输入：实验05剪枝 best.pt、configs/coco128_split.yaml
# 输出：runs/distill/experiment12_dinov2_distill_coco128/<时间戳>/
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe -m scripts.distill_dinov2_coco128 `
  --student "runs\prune\experiment05_greedy\20260908_161152\finetune\greedy\weights\best.pt" `
  --data "configs\coco128_split.yaml" `
  --epochs 30 `
  --batch 8 `
  --device 0 `
  --weight 0.5 `
  --warmup-epochs 2 `
  --dino-size 644
~~~

Linux Bash：

~~~bash
# 入口出处：scripts/distill_dinov2_coco128.py -> main()（第 510–593 行）
.venv/bin/python -m scripts.distill_dinov2_coco128 \
  --student runs/prune/experiment05_greedy/20260908_161152/finetune/greedy/weights/best.pt \
  --data configs/coco128_split.yaml \
  --epochs 30 \
  --batch 8 \
  --device 0 \
  --weight 0.5 \
  --warmup-epochs 2 \
  --dino-size 644
~~~

脚本顶部已有默认输入，也可以用 `--student`、`--data`、`--output` 显式覆盖。

### 10. 实验 12 结果

| 方案 | best mAP50-95 | last mAP50-95 |
|---|---:|---:|
| 输入剪枝模型 | 0.5962 | — |
| 普通微调 | 0.5962 | 0.4828 |
| P4 单尺度 DINOv2 | 0.5962 | **0.5040** |

- 蒸馏没有刷新原始 best。
- 单尺度 last 比普通微调 last 高 `0.0212`。
- 它缓解了后期退化，但没有创造新的峰值。
- best 与输入模型张量相同；last 有 90 个共同张量变化，说明训练确实执行。
- 验证集只有 26 张，本结果只能作为小规模方法验证。

## 四、实验 13：P3/P4/P5 多尺度 DINOv2 蒸馏

### 11. 单尺度怎样扩展为多尺度

多尺度仍只执行**一次教师前向**，用 `get_intermediate_layers()` 同时取三个 Transformer Block：

| DINOv2 Block | 从 0 开始的索引 | YOLO | 层权重 |
|---|---:|---|---:|
| 第 4 个 | 3 | P3 | 0.25 |
| 第 8 个 | 7 | P4 | 0.50 |
| 第 12 个 | 11 | P5 | 0.25 |

学生实际形状：

- P3：`[B, 40, 80, 80]`
- P4：`[B, 128, 40, 40]`
- P5：`[B, 256, 20, 20]`

三个尺度使用独立 projector，总 DINO 权重仍为 0.5。这样实验 12→13 主要只改变单/多尺度，方便归因。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
# 对应：MultiScaleDINOFeatureDistiller 第 143–185 行
# 输入：P3/P4/P5 三个学生特征；输出：加权总损失和三个分层损失

self.block_indices = (3, 7, 11)
self.level_weights = (0.25, 0.50, 0.25)
self.projectors = nn.ModuleList(
    nn.Conv2d(channels, 384, kernel_size=1, bias=False) for channels in student_channels
).to(device)


@torch.no_grad()
def teacher_features(self, images: torch.Tensor) -> tuple[torch.Tensor, ...]:
    images = F.interpolate(
        images,
        size=(self.dino_size, self.dino_size),
        mode="bilinear",
        align_corners=False,
    )
    normalized = (images - self.mean) / self.std
    return self.teacher.get_intermediate_layers(
        normalized,
        n=self.block_indices,
        reshape=True,
        norm=True,
    )


def loss(self, student_features: list[torch.Tensor], images: torch.Tensor):
    teacher_features = self.teacher_features(images)
    level_losses = []

    for student, teacher, projector in zip(
        student_features,
        teacher_features,
        self.projectors,
    ):
        teacher = F.interpolate(
            teacher,
            size=student.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        projected_student = F.normalize(projector(student), dim=1)
        normalized_teacher = F.normalize(teacher, dim=1)
        level_losses.append(
            1.0 - (projected_student * normalized_teacher).sum(dim=1).mean()
        )

    total = sum(
        weight * level_loss
        for weight, level_loss in zip(self.level_weights, level_losses)
    )
    return total, level_losses

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
#   _find_pyramid_layers()：第 273–278 行
#   attach_multiscale_distillation()：第 301–324 行
# 所属阶段：动态定位三个检测尺度、探测通道数、创建蒸馏器并注册 hook


def _find_pyramid_layers(model: nn.Module) -> list[nn.Module]:
    detect = model.model[-1]
    sources = getattr(detect, "f", None)
    if not isinstance(sources, (list, tuple)) or len(sources) != 3:
        raise RuntimeError("Could not identify Detect's P3/P4/P5 input layers")
    return [model.model[index] for index in sources]


def attach_multiscale_distillation(
    model: nn.Module,
    device: torch.device,
    dino_size: int,
    weight: float,
) -> None:
    captured: dict[str, torch.Tensor] = {}
    levels = ("p3", "p4", "p5")
    pyramid_layers = _find_pyramid_layers(model)
    probed: dict[str, torch.Tensor] = {}
    handles = []

    with torch.no_grad():
        model.eval()
        for level, layer in zip(levels, pyramid_layers):
            handles.append(
                layer.register_forward_hook(
                    lambda _module, _inputs, output, key=level: probed.__setitem__(
                        key, output
                    )
                )
            )
        model(torch.zeros(1, 3, 640, 640, device=device))
        for handle in handles:
            handle.remove()
        model.train()

    if any(level not in probed for level in levels):
        raise RuntimeError("P3/P4/P5 feature probe failed")

    channels = [probed[level].shape[1] for level in levels]
    model.add_module(
        "_dino_feature_distiller",
        MultiScaleDINOFeatureDistiller(channels, device, dino_size, weight),
    )
    model._dino_student_features = captured

    for level, layer in zip(levels, pyramid_layers):
        layer.register_forward_hook(CaptureFeature(captured, level))

    model.__class__ = MultiScaleDINOStudentModel

### 12. 实验 13 命令和结果

只比实验 12 多 `--multiscale`：

~~~powershell
# 入口出处：scripts/distill_dinov2_coco128.py -> main()（第 510–593 行）
# 输出：runs/distill/experiment13_dinov2_multiscale_coco128/<时间戳>/
Set-Location -LiteralPath "C:\Users\22565\OneDrive\Desktop\YOLO剪枝蒸馏初步尝试"
.\.venv\Scripts\python.exe -m scripts.distill_dinov2_coco128 `
  --student "runs\prune\experiment05_greedy\20260908_161152\finetune\greedy\weights\best.pt" `
  --data "configs\coco128_split.yaml" `
  --multiscale `
  --epochs 30 `
  --batch 8 `
  --device 0 `
  --weight 0.5 `
  --warmup-epochs 2 `
  --dino-size 644
~~~

~~~bash
# 入口出处：scripts/distill_dinov2_coco128.py -> main()（第 510–593 行）
.venv/bin/python -m scripts.distill_dinov2_coco128 \
  --student runs/prune/experiment05_greedy/20260908_161152/finetune/greedy/weights/best.pt \
  --data configs/coco128_split.yaml \
  --multiscale \
  --epochs 30 \
  --batch 8 \
  --device 0 \
  --weight 0.5 \
  --warmup-epochs 2 \
  --dino-size 644
~~~

| 方案 | best mAP50-95 | last mAP50-95 |
|---|---:|---:|
| 输入模型 | 0.5962 | — |
| 普通微调 | 0.5962 | 0.4828 |
| P4 单尺度 | 0.5962 | **0.5040** |
| P3/P4/P5 多尺度 | 0.5962 | 0.4957 |

多尺度比 control last 高 `0.0128`，但比单尺度低 `0.0084`。

### 13. 多尺度损失是否真的生效

| 损失 | epoch 1 | epoch 30 |
|---|---:|---:|
| 总 DINO loss | 0.9811 | 0.8986 |
| P3 | 1.0125 | 0.9814 |
| P4 | 0.9568 | 0.8722 |
| P5 | 0.9983 | 0.8687 |

三层都下降，说明 hook、教师前向、projector 和梯度链路都有效。P3 降幅最小，可能说明低层纹理和 DINO 中间语义较难对齐。

准确结论不是“多尺度没训练”，而是“多尺度训练生效，但没有转化成更高检测 mAP”。

## 五、为什么蒸馏没有提高 best（最重要的结果解释）

### 14. “best 不变”不等于“没有训练”

需要区分：

1. 原始剪枝 best 独立验证为 `0.5962`。
2. 训练中没有 epoch 稳定超过这个起点。
3. 实验 13 的训练 best 记录在约第 4 个 epoch；当时学生变化很小，EMA 以 FP16 保存后与输入权重在检查点精度上一致。
4. 后期 `last.pt` 有 90 个共同张量变化，但 mAP 更低。

所以正确表述：

> 蒸馏损失被优化、学生后期权重发生变化，但验证性能未刷新训练起点，因此部署 best 最终等价于输入剪枝模型。

可能原因：

- 102 张训练图太少，26 张验证图方差大。
- 学生已经是同数据上微调后的 best。
- DINOv2 是通用表征教师，不是检测教师。
- 当前 loss 对全图平均，背景干扰较强。
- projector 可以自己吸收部分对齐。
- P3/P5 与选取的 DINO 层可能不匹配。
- 检测梯度和蒸馏梯度可能冲突。
- 固定 BN、低学习率和关闭强增强提高稳定性，但也限制改变幅度。

## 六、训练权重与部署权重

### 15. 为什么训练后必须删除教师

训练 wrapper 包含 YOLO、DINOv2、projector、hook 和续训信息。部署权重只应包含 YOLO，否则文件变大、加载依赖 DINO 源码，还可能错误增加推理显存。

项目保存：

- `best_training_wrapper.pt` / `last_training_wrapper.pt`：研究与恢复训练。
- `best.pt` / `last.pt`：清理后的部署权重。

In [ ]:
# 规范化学习摘录：根据原项目整理，并非逐字复制
# 出处：scripts/distill_dinov2_coco128.py
#   detach_for_inference()：第 327–338 行
#   _write_clean_checkpoint()：第 381–386 行
# 所属阶段：删除训练专用对象，写出普通 YOLO 可加载的部署权重


def detach_for_inference(model: nn.Module) -> nn.Module:
    clean = copy.deepcopy(model).float().eval()

    for module in clean.modules():
        for hook_id, hook in list(module._forward_hooks.items()):
            if isinstance(hook, (CaptureP4, CaptureFeature)):
                del module._forward_hooks[hook_id]

    clean.__dict__.pop("_dino_student_features", None)
    if "_dino_feature_distiller" in clean._modules:
        del clean._modules["_dino_feature_distiller"]

    clean.__class__ = DetectionModel
    return clean


def _write_clean_checkpoint(wrapper_path: Path, clean_path: Path) -> None:
    checkpoint = torch.load(wrapper_path, map_location="cpu", weights_only=False)
    wrapped_model = checkpoint.get("ema") or checkpoint.get("model")
    checkpoint["ema"] = detach_for_inference(wrapped_model).half()
    checkpoint["model"] = None
    torch.save(checkpoint, clean_path)


# 正式脚本随后用 YOLO(str(clean_path)).val(...) 验证部署权重能够独立加载。

## 七、关键参数怎样改

### 16. 常用消融入口

**总蒸馏强度**

~~~powershell
# 参数出处：scripts/distill_dinov2_coco128.py -> main() 第 518 行
# 完整命令示例；其余参数继续使用脚本默认值
.\.venv\Scripts\python.exe -m scripts.distill_dinov2_coco128 --weight 0.25
~~~

可比较 `0.1 / 0.25 / 0.5 / 1.0`，每次只改这一项。

**warm-up**

~~~powershell
# 参数出处：scripts/distill_dinov2_coco128.py -> main() 第 519 行
.\.venv\Scripts\python.exe -m scripts.distill_dinov2_coco128 --warmup-epochs 5
~~~

**DINO 输入尺寸**

必须是 14 的倍数，如 `518=37×14`、`644=46×14`。越大越慢。

**教师层**

~~~python
# 出处：scripts/distill_dinov2_coco128.py
# 对应：MultiScaleDINOFeatureDistiller.__init__() 第 155 行
self.block_indices = (3, 7, 11)
~~~

索引从 0 开始，11 是第 12 个 Block。

**尺度权重**

~~~python
# 出处：scripts/distill_dinov2_coco128.py
# 对应：MultiScaleDINOFeatureDistiller.__init__() 第 156 行
self.level_weights = (0.25, 0.50, 0.25)
~~~

建议层权重和为 1，再用 `--weight` 控制总强度。

**损失形式**

可研究归一化 MSE、Smooth L1、attention transfer 或前景掩码，但必须保留 control，一次只改变一个变量。

In [ ]:
# 规范化研究建议代码：尚未在当前正式脚本中实现，不能当作已有实验结果
# 依据：scripts/distill_dinov2_coco128.py
#   多尺度加权：第 176–185 行
#   当前 cosine distance：第 134–140、179–184 行

# 方案 1：只调整现有多尺度权重；三个权重之和保持为 1
level_weights = (0.15, 0.70, 0.15)


def normalized_mse(
    projected_student: torch.Tensor,
    teacher_feature: torch.Tensor,
) -> torch.Tensor:
    """建议消融：师生先沿通道归一化，再计算 MSE。"""
    student = F.normalize(projected_student, dim=1)
    teacher = F.normalize(teacher_feature, dim=1)
    return F.mse_loss(student, teacher)


def foreground_weighted_cosine(
    projected_student: torch.Tensor,
    teacher_feature: torch.Tensor,
    foreground_mask: torch.Tensor,
) -> torch.Tensor:
    """建议消融：只定义损失；foreground_mask 的构造尚未在项目中实现。"""
    student = F.normalize(projected_student, dim=1)
    teacher = F.normalize(teacher_feature, dim=1)
    difference = 1.0 - (student * teacher).sum(dim=1)

    mask = foreground_mask.to(device=difference.device, dtype=difference.dtype)
    if mask.shape != difference.shape:
        raise ValueError(f"mask {mask.shape} must match loss map {difference.shape}")

    return (difference * mask).sum() / mask.sum().clamp_min(1.0)

## 八、输出目录与排错

### 17. 文件结构

~~~text
runs/distill/experiment13_dinov2_multiscale_coco128/<timestamp>/
├── train_config.json
├── run_info.json
├── comparison.csv
├── run.log
├── report.md
├── finetune/
│   ├── control/results.csv, results.png, weights/
│   └── distill/results.csv, results.png, weights/
└── validation/
    ├── source/
    ├── control/ 与 control_last/
    └── distill/ 与 distill_last/
~~~

看训练是否生效，优先检查 `results.csv` 的 DINO loss、各层 loss、学习率、best/last mAP；不要只看 best.pt。

### 18. 排错清单

- **device mismatch**：teacher、projector、mean/std buffer 和输入必须同设备。
- **hook 没抓到特征**：检查 Detect 的 `f` 和 P3/P4/P5 key。
- **教师被误训练**：确认 `teacher.training == False` 且参数 `requires_grad == False`。
- **DINO 尺寸错误**：`dino_size % 14 == 0`。
- **显存不足**：先减 batch，再减教师输入尺寸。
- **DINO loss 不下降**：检查 projector 是否进入 optimizer、权重是否大于 0、是否错误 detach。
- **loss 降而 mAP 不升**：可能是任务冲突或 projector 吸收，不代表代码必然错误。
- **部署加载失败**：确认已删除 teacher、hook、自定义模型类。
- **测速波动**：首次 CUDA 有预热成本；应多次测量取中位数。

## 九、可能追问的问题

### 19. 快速问答

**为什么不用 DINOv2 的分类软标签？**  
DINOv2 没有与 YOLO 同结构的 COCO 检测头，所以采用特征蒸馏。

**为什么冻结教师？**  
教师应是稳定目标；一起更新会让监督目标移动，并增加显存。

**为什么教师 644、YOLO 640？**  
ViT-S/14 patch size 为 14，644 可整除为 46×46 patch，再插值对齐。

**为什么 P4？**  
P4 在细节和语义间折中，与 DINO 最终层更容易匹配；实验也优于当前多尺度。

**为什么 1×1 Conv？**  
用较低开销解决通道数不同，不改变空间尺寸。

**为什么 cosine？**  
跨架构特征幅值不同，归一化后比较方向更稳定。

**为什么多尺度反而差？**  
更多约束不等于更准确；层级可能不匹配，梯度也可能与检测任务冲突。

**为什么 best 没变？**  
训练 epoch 没超过起点；早期微小变化经 EMA/FP16 保存后与输入一致，后期虽改变但精度更低。

**推理带不带 DINOv2？**  
不带。部署 best/last 已删除教师、projector 和 hook。

**能否证明 DINOv2 蒸馏无效？**  
不能。只能说明当前 COCO128、当前对齐和超参数没有提高 best。

## 十、结论与下一步

### 20. 当前阶段可以怎样总结

1. 建立了冻结 DINOv2 ViT-S/14 到剪枝 YOLO11n 的特征蒸馏流程。
2. 单尺度、多尺度均正常反向传播，部署模型不包含教师。
3. 两种方法都没有超过输入模型 `0.5962` 的 best。
4. last 对照说明蒸馏缓解了普通微调的后期退化。
5. 当前 P4 单尺度 last（`0.5040`）优于多尺度（`0.4957`）。
6. COCO128 只适合验证工程和提出假设，不能形成最终性能结论。

下一阶段优先级：

1. 先彻底读懂本 notebook 与脚本。
2. 在完整 COCO2017 上比较“无蒸馏 vs P4 单尺度”。
3. 若仍无提升，再研究前景掩码或真正的 YOLO 检测教师。
4. 固定数据划分、seed、训练设置，一次只改变一个因素。
5. 同时报告 best、last、训练曲线和多随机种子均值。

### 21. 建议学习顺序

- 第一遍：只看整体数据流和快速问答，能口头讲完整流程。
- 第二遍：在脚本中定位 `DINOFeatureDistiller`、`DINOStudentModel.loss`、`FrozenBNTrainer`。
- 第三遍：自己解释每个命令行参数，不必继续在 COCO128 大量跑参。
- 第四遍：从空白纸写出 `L_total = L_YOLO + λL_DINO`，说明每项由谁产生、谁接收梯度。
- 第五遍：练习解释“loss 下降但 mAP 不升”和“best/last 为什么不同”。

掌握标准：不看代码也能说明**输入、教师、学生、特征位置、尺寸与通道对齐、损失、梯度流向、命令、部署清理和实验局限**。